<a href="https://colab.research.google.com/github/visista19/30-days-challenge/blob/main/Mutli_agentwithCrewAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from crewai import Task,Agent,Crew

In [ ]:
!pip install langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 2.1 MB/s eta 0:00:00


In [ ]:
!pip install crewai


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 6.6 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of huggingface-hub to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of opentelemetry-exporter-otlp-proto-grpc to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of opentelemetry-exporter-otlp-proto-grpc to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of typer to determine which version is compatible 

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
!pip install litellm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 71.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 30.8 MB/s eta 0:00:00


In [ ]:
import os
from google.colab import userdata

# 1. Fetch your Groq API key from Colab Secrets
groq_key = userdata.get('GROK_API_KEY')

# 2. Redirect CrewAI's default OpenAI routing to Groq's infrastructure
os.environ["OPENAI_API_BASE"] = "https://api.groq.com/openai/v1"
os.environ["OPENAI_API_KEY"] = groq_key
os.environ["OPENAI_MODEL_NAME"] = "llama-3.3-70b-versatile"

In [ ]:
planner = Agent(
    role="Content Planner",
    goal="Plan engaging and factually accurate content on {topic}",
    backstory="You're working on planning a blog article "
              "about the topic: {topic}."
              "You collect information that helps the "
              "audience learn something "
              "and make informed decisions. "
              "Your work is the basis for "
              "the Content Writer to write an article on this topic.",
    allow_delegation=False,
	  verbose=True
)

In [ ]:
writer = Agent(
    role="Content Writer",
    goal="Write insightful and factually accurate "
         "opinion piece about the topic: {topic}",
    backstory="You're working on a writing "
              "a new opinion piece about the topic: {topic}. "
              "You base your writing on the work of "
              "the Content Planner, who provides an outline "
              "and relevant context about the topic. "
              "You follow the main objectives and "
              "direction of the outline, "
              "as provide by the Content Planner. "
              "You also provide objective and impartial insights "
              "and back them up with information "
              "provide by the Content Planner. "
              "You acknowledge in your opinion piece "
              "when your statements are opinions "
              "as opposed to objective statements.",
    allow_delegation=False,
    verbose=True
)

In [ ]:
editor = Agent(
    role="Editor",
    goal="Edit a given blog post to align with "
         "the writing style of the organization. ",
    backstory="You are an editor who receives a blog post "
              "from the Content Writer. "
              "Your goal is to review the blog post "
              "to ensure that it follows journalistic best practices,"
              "provides balanced viewpoints "
              "when providing opinions or assertions, "
              "and also avoids major controversial topics "
              "or opinions when possible.",
    allow_delegation=False,
    verbose=True
)

In [ ]:
Plan = Task(
    description=(
        "1. Prioritize the latest trends, key players, "
            "and noteworthy news on {topic}.\n"
        "2. Identify the target audience, considering "
            "their interests and pain points.\n"
        "3. Develop a detailed content outline including "
            "an introduction, key points, and a call to action.\n"
        "4. Include SEO keywords and relevant data or sources."
    ), expected_output="A comprehensive content plan document "
        "with an outline, audience analysis, "
        "SEO keywords, and resources.",
    agent=planner,
    )


In [ ]:
write = Task(
    description=(
        "1. Use the content plan to craft a compelling "
            "blog post on {topic}.\n"
        "2. Incorporate SEO keywords naturally.\n"
		"3. Sections/Subtitles are properly named "
            "in an engaging manner.\n"
        "4. Ensure the post is structured with an "
            "engaging introduction, insightful body, "
            "and a summarizing conclusion.\n"
        "5. Proofread for grammatical errors and "
            "alignment with the brand's voice.\n"
    ),
    expected_output="A well-written study material "
        "in markdown format, ready for publication, "
        "each section should have 2 or 3 paragraphs.",
    agent=writer,
)

In [ ]:
edit = Task(
    description=("Proofread the given study material for "
                 "grammatical errors and "
                 "alignment with the brand's voice."),
    expected_output="A well-written study material in markdown format, "
                    "ready for publication, "
                    "each section should have 2 or 3 paragraphs.",
    agent=editor
)

In [ ]:
crew = Crew(
    agents = [planner, writer, editor],
    tasks=[Plan, write, edit],
    verbose=True
)

In [ ]:
result = await crew.kickoff_async(inputs={"topic": "UPSC EXAMS"})

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 109c7b3c-468a-4e58-b4d6-97349dac9c83                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: 1. Prioritize the latest trends, key players, and noteworthy news on UPSC EXAMS.                         │
│  2. Identify the target audience, considering their interests and pain points.                                  │
│  3. Develop a detailed content outline including an introduction, key points, and a call to action.             │
│  4. Include SEO keywords and relevant data or sources.                                                          │
│  ID: 07b70f13-fe70-48d2-8919-114456b4738a                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Planner                                                                                         │
│                                                                                                                 │
│  Task: 1. Prioritize the latest trends, key players, and noteworthy news on UPSC EXAMS.                         │
│  2. Identify the target audience, considering their interests and pain points.                                  │
│  3. Develop a detailed content outline including an introduction, key points, and a call to action.             │
│  4. Include SEO keywords and relevant data or sources.                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Planner                                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Comprehensive Content Plan Document: UPSC EXAMS**                                                            │
│                                                                                                                 │
│  **I. Introduction**                                                                                            │
│                                                                                                                 │
│  The Union Public Service Commission (UPSC) exams are a series of competitive examinations conducted by the     │
│  Indian government to select candidates for various civil service positions. The exams are highly competitive,  │
│  with thousands of candidates vying for a limited number of positions. This article aims to provide an          │
│  overview of the UPSC exams, including the latest trends, key players, and noteworthy news, as well as offer    │
│  guidance and support to candidates preparing for the exams.                                                    │
│                                                                                                                 │
│  **II. Target Audience Analysis**                                                                               │
│                                                                                                                 │
│  The target audience for this article consists of individuals interested in appearing for the UPSC exams,       │
│  including:                                                                                                     │
│                                                                                                                 │
│  * Students and graduates aged 21-30                                                                            │
│  * Working professionals looking to transition to a career in civil service                                     │
│  * Individuals who have previously attempted the UPSC exams and are seeking guidance on how to improve their    │
│  performance                                                                                                    │
│                                                                                                                 │
│  The target audience is likely to be interested in the following topics:                                        │
│                                                                                                                 │
│  * UPSC exam syllabus and pattern                                                                               │
│  * Preparation strategies and tips                                                                              │
│  * Current events and news related to the UPSC exams                                                            │
│  * Success stories and experiences of previous candidates                                                       │
│                                                                                                                 │
│  Pain points of the target audience:                                                                            │
│                                                                                                                 │
│  * Limited understanding of the UPSC exam format and sy

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 1. Prioritize the latest trends, key players, and noteworthy news on UPSC EXAMS.                         │
│  2. Identify the target audience, considering their interests and pain points.                                  │
│  3. Develop a detailed content outline including an introduction, key points, and a call to action.             │
│  4. Include SEO keywords and relevant data or sources.                                                          │
│  Agent: Content Planner                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (7 outputs left out)

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Editor                                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### Introduction to UPSC Exams                                                                                 │
│  The Union Public Service Commission (UPSC) exams are a series of highly competitive examinations conducted by  │
│  the Indian government to select candidates for various civil service positions. These exams are considered     │
│  among the most challenging in the country, with thousands of candidates vying for a limited number of          │
│  positions. The significance of the UPSC exams lies in the opportunities they provide for candidates to serve   │
│  the nation in various capacities, ranging from administrative roles to diplomatic positions.                   │
│                                                                                                                 │
│  The history and evolution of the UPSC exams are marked by continuous efforts to improve the selection          │
│  process, ensuring that the most capable and dedicated individuals are chosen for these esteemed roles. The     │
│  UPSC conducts various exams, including the Civil Services Examination, the Indian Forest Service Examination,  │
│  and others, each serving as a gateway to different career paths within the civil services. Understanding the   │
│  types of UPSC exams and their specific requirements is crucial for candidates who aspire to join the civil     │
│  services.                                                                                                      │
│                                                                                                                 │
│  The UPSC exams have undergone significant changes over the years, with updates to the syllabus and pattern     │
│  aimed at assessing the candidates' abilities more comprehensively. For instance, the inclusion of more         │
│  contemporary issues and themes in the exam syllabus reflects the evolving nature of governance and public      │
│  policy. It is essential for candidates to stay abreast of these developments to remain competitive.            │
│                                                                                                                 │
│  ### Latest Trends and News                                                                                     │
│  Recent years have seen several key changes to the UPSC exam syllabus and pattern. One of the notable updates   │
│  is the introduction of new subjects and the reorientation of existing ones to better align with the current    │
│  needs of the civil services. The UPSC has also made efforts to increase transparency and fairness in the       │
│  selection process, including the introduction of more objective evaluation methods and stricter norms against  │
│  malpractices.                                                                                                  │
│                                                                                                                 │
│  Noteworthy news and events related to the UPSC exams often provide valuable insights for candidates. Updates   │
│  on the government's recruitment policies, changes in the exam format, and success stories of previous          │
│  candidates can serve as motivation and guidance. For example, the government's initiatives to increase the     │
│  representation of underrepresented groups in the civil

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Proofread the given study material for grammatical errors and alignment with the brand's voice.          │
│  Agent: Editor                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [ ]:
from IPython.display import Markdown
Markdown(result.raw)

### Introduction to UPSC Exams
The Union Public Service Commission (UPSC) exams are a series of highly competitive examinations conducted by the Indian government to select candidates for various civil service positions. These exams are considered among the most challenging in the country, with thousands of candidates vying for a limited number of positions. The significance of the UPSC exams lies in the opportunities they provide for candidates to serve the nation in various capacities, ranging from administrative roles to diplomatic positions.

The history and evolution of the UPSC exams are marked by continuous efforts to improve the selection process, ensuring that the most capable and dedicated individuals are chosen for these esteemed roles. The UPSC conducts various exams, including the Civil Services Examination, the Indian Forest Service Examination, and others, each serving as a gateway to different career paths within the civil services. Understanding the types of UPSC exams and their specific requirements is crucial for candidates who aspire to join the civil services.

The UPSC exams have undergone significant changes over the years, with updates to the syllabus and pattern aimed at assessing the candidates' abilities more comprehensively. For instance, the inclusion of more contemporary issues and themes in the exam syllabus reflects the evolving nature of governance and public policy. It is essential for candidates to stay abreast of these developments to remain competitive. 

### Latest Trends and News
Recent years have seen several key changes to the UPSC exam syllabus and pattern. One of the notable updates is the introduction of new subjects and the reorientation of existing ones to better align with the current needs of the civil services. The UPSC has also made efforts to increase transparency and fairness in the selection process, including the introduction of more objective evaluation methods and stricter norms against malpractices.

Noteworthy news and events related to the UPSC exams often provide valuable insights for candidates. Updates on the government's recruitment policies, changes in the exam format, and success stories of previous candidates can serve as motivation and guidance. For example, the government's initiatives to increase the representation of underrepresented groups in the civil services have led to the introduction of special provisions and incentives.

Staying informed about such developments can help candidates navigate the preparation process more effectively and make the most of the opportunities available to them. The UPSC's official website and other reputable sources provide regular updates on the exams, including notifications, syllabus changes, and results. Candidates should regularly check these sources to stay updated and plan their preparation accordingly.

### Preparation Strategies and Tips
Preparation for the UPSC exams is a demanding and intensive process that requires careful planning, dedication, and persistence. The first step in preparing for the exams is to gain a thorough understanding of the exam syllabus and pattern. This involves familiarizing oneself with the subjects, topics, and types of questions that are typically asked. The UPSC exam syllabus is comprehensive and covers a wide range of subjects, including history, geography, economy, politics, and science, among others.

Recommended study materials and resources play a critical role in preparation. Candidates should rely on a combination of standard textbooks, reference materials, and online resources to cover the vast syllabus. Additionally, practicing with previous years' question papers and mock tests is essential for assessing one's knowledge, identifying weaknesses, and improving time management skills.

Time management and planning strategies are also vital, as candidates need to balance the preparation of various subjects and topics within a limited timeframe. Creating a schedule and sticking to it, setting achievable milestones, and regularly reviewing progress can help in staying focused and motivated.

### Key Players and Success Stories
The journey to success in the UPSC exams is often facilitated by key players such as coaching institutes, mentors, and successful candidates who have navigated the process before. Coaching institutes and online resources offer structured guidance, study materials, and practice tests that can significantly enhance a candidate's preparation. Many successful candidates attribute their success to the support and strategy provided by these institutions.

Success stories of previous candidates serve as powerful motivation and provide practical lessons for aspirants. These stories often highlight the importance of perseverance, hard work, and the right strategy. They also demonstrate that success in the UPSC exams is achievable through dedicated effort, regardless of one's background or circumstances.

The role of coaching institutes and online resources in preparation cannot be overstated. They provide a structured approach to studying, which is particularly beneficial for candidates who may not have access to quality educational resources or guidance.

### Conclusion
The UPSC exams are a challenging and competitive process, but with the right guidance, support, and mindset, candidates can significantly increase their chances of success. By understanding the exam syllabus and pattern, staying updated with the latest trends and news, following a structured preparation strategy, and learning from the experiences of others, candidates can overcome the obstacles and achieve their goals.

In conclusion, preparing for the UPSC exams is a journey that requires dedication, resilience, and a deep commitment to public service. While the path can be daunting, the rewards of serving the nation and contributing to its development are immeasurable. As candidates embark on this journey, it is essential to stay focused, adapt to challenges, and continuously learn and grow. The UPSC exams are not just a test of knowledge but also of character, integrity, and the ability to lead and serve. 

To all the aspirants, the journey to success in the UPSC exams begins with a dream, is nurtured by hard work, and is fulfilled by perseverance. It is a path that demands sacrifice, patience, and an unwavering commitment to one's goals. The fulfillment and sense of purpose that come from serving the nation and its people make every effort worthwhile. As you prepare for the exams, remember that you are not just competing for a position but are also being groomed to be leaders and change-makers of tomorrow.

### Additional Guidance
Staying motivated and focused throughout the preparation period is crucial. Candidates should develop strategies to maintain their mental and physical well-being, such as engaging in regular exercise, meditation, or hobbies, and seeking support from family, friends, or peers. Furthermore, staying updated with current events and news is crucial, as it not only helps in preparing for the exams but also broadens one's perspective and understanding of the world.

In the end, success in the UPSC exams is not just about achieving a personal goal but also about contributing to the nation's development and serving its people. With the right approach, mindset, and support, any aspirant can overcome the challenges and achieve their dreams. 

### Final Thoughts
The UPSC exams are a prestigious and highly competitive process, but they also offer a unique opportunity for candidates to serve the nation and make a meaningful difference in the lives of others. As candidates prepare for the exams, they should remember that the journey is just as important as the destination. The knowledge, skills, and values gained during the preparation period will not only help them succeed in the exams but also shape them into capable and compassionate leaders.

By embracing the challenges and opportunities presented by the UPSC exams, candidates can grow both personally and professionally, and develop a deeper understanding of themselves and their role in society. The exams are not just a test of knowledge but also a transformative experience that can help candidates discover their potential, build their character, and develop a sense of purpose and direction. 

### Last Reminder
As you prepare for the UPSC exams, remember to stay focused, motivated, and committed to your goals. Believe in yourself and your abilities, and do not be afraid to seek help and guidance when needed. The journey to success in the UPSC exams is long and challenging, but with the right mindset, strategy, and support, you can overcome any obstacle and achieve your dreams. Good luck!